# Turb-DETR — Underwater Plastic Detection (Colab Training)

End-to-end training pipeline using **RT-DETR** via Ultralytics on underwater trash datasets.

| Item | Detail |
|---|---|
| Model | RT-DETR-L (COCO pretrained) |
| Datasets | Trash-ICRA19, UFO-120, TrashCan, RUIE |
| Framework | Ultralytics 8.3+ / PyTorch 2.2+ |
| Runtime | GPU (T4 or better) |

> **Before running:** Go to *Runtime → Change runtime type → **GPU*** 

## 1. Environment Setup

In [ ]:
# ── 1a. Check GPU availability ──────────────────────────────
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

In [ ]:
# ── 1b. Set up repository ────────────────────────────────────
# OPTION A (default): Clone from GitHub.
#   Requires the repo to be pushed to GitHub first.
#   Push your local code: git push origin main
#
# OPTION B: Upload your repo folder to Google Drive, then:
#   import shutil
#   shutil.copytree("/content/drive/MyDrive/turb-detr-underwater-detection", REPO_DIR)
#
# Currently using OPTION A. If clone fails, switch to OPTION B.

import os

REPO_URL = "https://github.com/csdeepak/turb-detr-underwater-detection.git"
REPO_DIR = "/content/turb-detr-underwater-detection"

if not os.path.exists(REPO_DIR):
    result = os.system(f"git clone {REPO_URL} {REPO_DIR}")
    if result != 0:
        print("=" * 60)
        print("⚠  git clone FAILED.")
        print("   The GitHub repo may not be public or doesn't exist yet.")
        print("   OPTION B: upload the project folder to MyDrive and run:")
        print("     import shutil")
        print("     shutil.copytree(")
        print('       "/content/drive/MyDrive/turb-detr-underwater-detection",')
        print(f'       "{REPO_DIR}")')
        print("=" * 60)
    else:
        print(f"✓ Repository cloned to {REPO_DIR}")
else:
    print(f"✓ Repository already exists at {REPO_DIR}")

%cd {REPO_DIR}

In [ ]:
# ── 1c. Install project dependencies ────────────────────────
!pip install -q -r requirements.txt

In [ ]:
# ── 1d. Verify installation & GPU ───────────────────────────
import torch
import ultralytics

ultralytics.checks()  # prints ultralytics + system info

print(f"\nPyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"Device   : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Ultralytics : {ultralytics.__version__}")

## 2. Datasets

**How this works:**
- Drive is mounted first (cell 2a)
- Each dataset cell will pause and ask you to paste the Google Drive URL
- Paste the URL and press Enter — it downloads and restructures automatically

In [ ]:
# ── 2a. Mount Google Drive ──────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ── 2b. Install gdown ────────────────────────────────────────
!pip install -q gdown
print("gdown ready.")

In [ ]:

# ── 2c. Download & restructure Trash-ICRA19 ─────────────────
# Paste the Google Drive folder URL when prompted.
# URL format: https://drive.google.com/drive/folders/FOLDER_ID
#
# Fix: upgrades gdown, uses authenticated cookies, and falls back
# to downloading only the dataset/ subfolder to skip large weight files.

import subprocess, importlib, re, shutil, os
from pathlib import Path

# ── Step 0: upgrade gdown (fixes JSONDecodeError on large folders) ──
print("Upgrading gdown …")
subprocess.run(["pip", "install", "-qU", "gdown"], check=True)

import gdown
importlib.reload(gdown)
print(f"gdown {gdown.__version__} ready.\n")

# ── Step 1: ask for URL ──────────────────────────────────────
url = input("\nPaste Google Drive URL for trash_icra19 folder: ").strip()

folder_id_match = re.search(r'/folders/([a-zA-Z0-9_-]+)', url)
if not folder_id_match:
    raise ValueError(f"Could not extract folder ID from URL: {url}")

folder_id = folder_id_match.group(1)
raw_dir   = Path("data/_trash_icra19_raw")
raw_dir.mkdir(parents=True, exist_ok=True)

# ── Step 2: download (with progressive fallback) ─────────────
print(f"Downloading folder ID: {folder_id} …")

def _try_folder(fid, dest, label=""):
    """Try gdown.download_folder; return True on success."""
    try:
        gdown.download_folder(
            id=fid,
            output=str(dest),
            quiet=False,
            use_cookies=True,       # uses your Colab Google auth
            remaining_ok=True,      # tolerate truncated large folders
        )
        return True
    except Exception as exc:
        print(f"[WARN] {label} folder download failed: {exc}")
        return False

# Attempt 1 – full folder
if not _try_folder(folder_id, raw_dir, "Full"):
    # Attempt 2 – look for a 'dataset' sub-folder in the root listing
    # (some Trash-ICRA19 Drive mirrors nest images inside dataset/)
    print("\nAttempting partial download: listing root folder …")
    try:
        files = gdown.download_folder(
            id=folder_id,
            output=str(raw_dir / "_tmp_list"),
            quiet=True,
            use_cookies=True,
            skip_download=True,     # list only, do not download files
        )
    except Exception:
        files = []

    # Try known sub-folder IDs extracted from the Drive link tree
    sub_candidates = []
    if files:
        for f in files:
            if hasattr(f, "name") and f.name.lower() in ("dataset", "images", "train"):
                sub_candidates.append(f.id)

    if sub_candidates:
        for sub_id in sub_candidates:
            if _try_folder(sub_id, raw_dir / "dataset", f"sub-folder {sub_id}"):
                break
    else:
        # Last resort: download files in the root only (no recursion)
        print("[WARN] Could not resolve sub-folders automatically.")
        print("       Try running:  gdown --folder <URL>  in a terminal cell,")
        print("       then re-run the restructuring steps below manually.")

# ── Step 3: show what arrived ────────────────────────────────
print("\nDownloaded structure:")
for root, dirs, files in os.walk(raw_dir):
    lvl = root.replace(str(raw_dir), "").count(os.sep)
    indent = "  " * lvl
    print(f"{indent}{Path(root).name}/")
    if lvl < 4:
        for f in sorted(files)[:4]:
            print(f"{indent}  {f}")
        if len(files) > 4:
            print(f"{indent}  ... ({len(files)} files)")

# ── Step 4: find the split root ──────────────────────────────
def find_split_root(base: Path) -> Path | None:
    """Find the directory that directly contains train/ and val/ folders."""
    for candidate in [base] + list(base.rglob("*")):
        if candidate.is_dir():
            children = {p.name.lower() for p in candidate.iterdir() if p.is_dir()}
            if "train" in children and "val" in children:
                return candidate
    return None

split_root = find_split_root(raw_dir)
if split_root is None:
    print("\n✗ Could not find train/ and val/ directories in the download.")
    print("  The folder structure may be unexpected. Contents above.")
    raise RuntimeError("Dataset restructure failed — check output above.")

print(f"\n✓ Split root: {split_root}")

# ── Step 5: restructure to YOLO format ───────────────────────
out_root = Path("data/trash_icra19")
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

for split_dir in sorted(split_root.iterdir()):
    if not split_dir.is_dir():
        continue
    split = split_dir.name.lower()           # train / val / test
    if split not in ("train", "val", "test"):
        continue

    imgs_dst = out_root / "images" / split
    lbls_dst = out_root / "labels" / split
    imgs_dst.mkdir(parents=True, exist_ok=True)
    lbls_dst.mkdir(parents=True, exist_ok=True)

    n_imgs = n_lbls = 0
    for f in split_dir.rglob("*"):           # handles flat OR nested sub-dirs
        if f.is_file():
            if f.suffix.lower() in IMAGE_EXTS:
                shutil.copy2(f, imgs_dst / f.name)
                n_imgs += 1
            elif f.suffix.lower() == ".txt" and f.stem.lower() != "readme":
                shutil.copy2(f, lbls_dst / f.name)
                n_lbls += 1

    print(f"  {split:6s} → {n_imgs:>5} images  |  {n_lbls:>5} labels")

print(f"\n✓ Dataset ready at {out_root}")
print("→ Run cell 3a next to verify.")


## 3. Verify Dataset Structure

In [ ]:
# ── 3a. Verify expected folder structure per dataset ─────────
from pathlib import Path

EXPECTED_SPLITS = ["train", "val", "test"]

for ds_name in ["trash_icra19", "ufo120", "trashcan", "ruie"]:
    ds_root = Path("data") / ds_name
    print(f"\n{'─'*50}")
    print(f"📂 {ds_name}  →  {'EXISTS' if ds_root.exists() else '⚠ MISSING'}")
    if not ds_root.exists():
        continue

    # Check for images/ and labels/ subdirectories per split
    for split in EXPECTED_SPLITS:
        img_dir = ds_root / "images" / split
        lbl_dir = ds_root / "labels" / split
        img_ok = img_dir.exists()
        lbl_ok = lbl_dir.exists()
        n_imgs = len(list(img_dir.glob("*"))) if img_ok else 0
        n_lbls = len(list(lbl_dir.glob("*"))) if lbl_ok else 0
        status = "✓" if (img_ok and lbl_ok) else "✗"
        print(f"  {status} {split:6s} — images: {n_imgs:>5,}  labels: {n_lbls:>5,}")

In [ ]:
# ── 3b. Print dataset statistics ─────────────────────────────
from pathlib import Path
from collections import Counter

def dataset_stats(ds_path: Path, class_names: list[str] | None = None):
    """Print image counts and class distribution for a YOLO-format dataset."""
    print(f"\n{'='*55}")
    print(f"  Dataset: {ds_path.name}")
    print(f"{'='*55}")

    total_imgs, total_objects = 0, 0
    class_counts = Counter()

    for split in ["train", "val", "test"]:
        img_dir = ds_path / "images" / split
        lbl_dir = ds_path / "labels" / split
        n_imgs = len(list(img_dir.glob("*"))) if img_dir.exists() else 0
        total_imgs += n_imgs

        if lbl_dir.exists():
            for lbl_file in lbl_dir.glob("*.txt"):
                for line in lbl_file.read_text().strip().splitlines():
                    parts = line.strip().split()
                    if parts:
                        class_counts[int(parts[0])] += 1
                        total_objects += 1

        print(f"  {split:6s}: {n_imgs:>6,} images")

    print(f"  {'─'*40}")
    print(f"  Total : {total_imgs:>6,} images, {total_objects:>7,} objects")

    if class_counts:
        print(f"\n  Class distribution:")
        for cls_id in sorted(class_counts):
            name = class_names[cls_id] if class_names and cls_id < len(class_names) else f"class_{cls_id}"
            pct = class_counts[cls_id] / total_objects * 100
            bar = "█" * int(pct / 2)
            print(f"    {cls_id}: {name:15s} {class_counts[cls_id]:>6,}  ({pct:5.1f}%) {bar}")

# Run stats for Trash-ICRA19 (primary dataset)
trash_icra19_classes = ["plastic", "bottle", "can", "bag", "net"]
ds = Path("data/trash_icra19")
if ds.exists():
    dataset_stats(ds, trash_icra19_classes)
else:
    print("⚠ data/trash_icra19 not found — check symlinks above.")

## 4. Train Models

**Run 4a (baseline) first**, then 4b (Turb-DETR).
Both must complete before evaluation and ablation are meaningful.

| Run | Model | SimAM | Purpose |
|---|---|---|---|
| 4a | RT-DETR (baseline) | ✗ | Comparison baseline |
| 4b | Turb-DETR | ✓ | Your architecture |

In [ ]:
# ── 4a. Train BASELINE RT-DETR (no SimAM) ───────────────────
# This is your controlled baseline for comparison.
# use_simam=False → vanilla RT-DETR, no architecture change.
import sys
sys.path.insert(0, "/content/turb-detr-underwater-detection")

from models.turb_detr import TurbDETR

baseline = TurbDETR(model_variant="rtdetr-l", use_simam=False)
baseline.info()

baseline_results = baseline.train(
    data_cfg="configs/trash_icra19.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=1e-4,
    lrf=0.01,
    weight_decay=1e-4,
    warmup_epochs=3,
    patience=15,
    amp=True,
    workers=4,
    project="outputs",
    name="rtdetr_baseline",
    save_period=10,
    exist_ok=True,
    verbose=True,
)

print("\nBaseline training complete.")
print(f"mAP@0.5      : {baseline_results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
print(f"mAP@0.5:0.95 : {baseline_results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")

In [ ]:
# ── 4b. Train TURB-DETR (with SimAM) ────────────────────────
# Your architecture: RT-DETR + SimAM turbidity suppression.
# This is the model you are comparing against the baseline above.

from models.turb_detr import TurbDETR

turb_detr = TurbDETR(model_variant="rtdetr-l", use_simam=True)
turb_detr.info()

turb_results = turb_detr.train(
    data_cfg="configs/trash_icra19.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=1e-4,
    lrf=0.01,
    weight_decay=1e-4,
    warmup_epochs=3,
    patience=15,
    amp=True,
    workers=4,
    project="outputs",
    name="turb_detr",
    save_period=10,
    exist_ok=True,
    verbose=True,
)

print("\nTurb-DETR training complete.")
print(f"mAP@0.5      : {turb_results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
print(f"mAP@0.5:0.95 : {turb_results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")

## 5. Evaluate & Visualize

In [ ]:
# ── 5a. Compare baseline vs Turb-DETR on test set ───────────
from pathlib import Path

BASELINE_WEIGHTS  = "outputs/rtdetr_baseline/weights/best.pt"
TURBDETR_WEIGHTS  = "outputs/turb_detr/weights/best.pt"

results_table = []

for name, weights, use_simam in [
    ("RT-DETR baseline",  BASELINE_WEIGHTS, False),
    ("Turb-DETR (SimAM)", TURBDETR_WEIGHTS, True),
]:
    wpath = Path(weights)
    if not wpath.exists():
        print(f"⚠ {name}: weights not found at {weights} — skip")
        continue

    from models.turb_detr import TurbDETR
    m = TurbDETR(weights=weights, use_simam=use_simam)
    metrics = m.validate(data_cfg="configs/trash_icra19.yaml", split="test")

    row = {
        "Model":     name,
        "mAP@0.5":   metrics.box.map50,
        "mAP@0.5:95":metrics.box.map,
        "Precision": metrics.box.mp,
        "Recall":    metrics.box.mr,
    }
    results_table.append(row)
    del m  # free GPU memory

print(f"\n{'Model':<24} {'mAP@0.5':>9} {'mAP@.5:.95':>12} {'Prec':>8} {'Recall':>8}")
print("─" * 64)
for r in results_table:
    print(f"{r['Model']:<24} {r['mAP@0.5']:>9.4f} {r['mAP@0.5:95']:>12.4f} {r['Precision']:>8.4f} {r['Recall']:>8.4f}")

# ── 5b. Inference on sample test images ──────────────────────
from IPython.display import Image, display
from pathlib import Path
import sys, glob

sys.path.insert(0, "/content/turb-detr-underwater-detection")
from models.turb_detr import TurbDETR

# Load best Turb-DETR weights for inference
TURBDETR_WEIGHTS = "outputs/turb_detr/weights/best.pt"
if not Path(TURBDETR_WEIGHTS).exists():
    print(f"⚠ {TURBDETR_WEIGHTS} not found — run training cells first.")
else:
    infer_model = TurbDETR(weights=TURBDETR_WEIGHTS, use_simam=True)

    test_dir     = Path("data/trash_icra19/images/test")
    sample_imgs  = sorted(glob.glob(str(test_dir / "*")))[:4]

    if not sample_imgs:
        print(f"⚠ No images found in {test_dir}")
    else:
        preds = infer_model.predict(
            source=sample_imgs,
            imgsz=640,
            conf=0.25,
            save=True,
            project="outputs",
            name="turb_detr_predictions",
            exist_ok=True,
        )
        pred_dir = Path("outputs/turb_detr_predictions")
        for img_path in sorted(pred_dir.glob("*.jpg"))[:4]:
            display(Image(filename=str(img_path), width=640))
        del infer_model

In [ ]:
# ── 5c. Save all weights to Google Drive ─────────────────────
import shutil
from pathlib import Path

dst = Path("/content/drive/MyDrive/turb_detr_weights/")
dst.mkdir(parents=True, exist_ok=True)

weights_to_save = {
    "rtdetr_baseline_best.pt":  "outputs/rtdetr_baseline/weights/best.pt",
    "rtdetr_baseline_last.pt":  "outputs/rtdetr_baseline/weights/last.pt",
    "turb_detr_best.pt":        "outputs/turb_detr/weights/best.pt",
    "turb_detr_last.pt":        "outputs/turb_detr/weights/last.pt",
}

for dst_name, src_path in weights_to_save.items():
    src = Path(src_path)
    if src.exists():
        shutil.copy2(src, dst / dst_name)
        print(f"  ✓ {dst_name:<35} {src.stat().st_size / 1e6:.1f} MB")
    else:
        print(f"  ✗ {dst_name:<35} not found (training may not have completed)")

print(f"\nAll available weights saved to {dst}")